# v2 REAL — 개인 의도 보정과 XYZ 배율

기존 strict JSON worker를 사용합니다. 처음에는 DRY RUN이고, PC에서 REAL RUN을 선택하면 실제 명령이 전송됩니다.

위치 매핑: `operator delta → C → XYZ gains → 20cm 작업범위 → IK`. Z를 고정하거나 전진 궤적으로 강제 투영하지 않습니다.
기준 자세는 로봇의 현재 EEF와 편한 컨트롤러 자세를 양손 grip 시점에 대응시킵니다. 좌표는 로봇 전방 +X, 왼쪽 +Y, 위 +Z입니다.


**자세 유지 IK 적용:** 첫 양손 grip 조작의 실측 자세를 기준으로 유지합니다(DRY RUN 포함). 위치·방향 허용오차는 1 mm·0.5°이며, grip을 다시 잡아도 자세 기준은 유지됩니다. 기존 IK 비교는 `POSTURE_IK_ENABLED=False`로 바꾸고 런타임을 다시 시작하세요. [설정·테스트 안내](../docs/POSTURE_IK.md).


수동 관절 조정: 조작을 멈춘 뒤 관절과 절대 목표 각도(°)를 선택하고 **관절 미리보기 → 조정 실행**을 누릅니다. 같은 창의 **Cancel return**으로 중단하고, 조정 후 **Initial pose**로 복귀할 수 있습니다. 최초 HOME 목표는 보존됩니다.
그리퍼는 설정 셀의 60°/s 상한을 사용하며 실행 창의 속도 입력은 없습니다. REAL 적용 전 로봇 PC의 `inbound.py`와 `fixed_quest_protocol.py`를 동일 버전으로 교체하고 inbound/outbound를 재시작하세요. HOME과 수동 조정은 최대 10°/s입니다.


In [ ]:
# Locate this standalone workspace from a notebook or repository directory.
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]


import numpy as np


## 설정 — 시작 보정과 제어 설정

기존 순서대로 설정 → 모델 준비 → 실행 셀을 실행하세요. 노트북에는 선택 위젯을 표시하지 않습니다. 시작 보정은 아래 `PROFILE_PATH`이며 기본값은 기존 v1입니다.

MuJoCo 커맨드 센터의 **작업자 보정** 목록에서 **보정 미적용 (identity)**, **v1 원본 (v1.json)**, 또는 작업자 파일명을 선택하면 자동으로 적용합니다. `calibration/profiles/`와 `calibration/diagnostics/`의 보정 프로필 JSON을 표시합니다. 새 파일은 **새로고침**으로 찾고 적용된 파일명·경로·profile ID를 확인하세요. v1은 재계산 없이 원본 행렬을 사용합니다.

**DRY RUN 대기 상태에서는 추적·grip 확인 없이 XYZ 배율과 보정 프로필을 변경할 수 있습니다.** 변경 후 REAL RUN으로 전환해도 적용값은 유지됩니다. REAL RUN에서 변경할 때는 기존처럼 양손 추적이 유효하고 grip을 모두 놓아야 합니다. 미리보기 조작·자동 복귀 중 변경 차단과 녹화 중 프로필 선택 잠금은 유지합니다. 잘못된 파일은 적용하지 않고 현재 보정을 유지하며 오류를 표시합니다. 설정 셀을 다시 실행하면 그 셀의 `PROFILE_PATH`로 초기화됩니다. SIM 확인 기록이나 setup 확인 옵션은 필요하지 않습니다.

XY 0.5·Z 0.8, 팔 속도 상한 60°/s, 그리퍼 상한 60°/s, Initial pose 10°/s입니다. 팔 속도 입력·적용 UI는 유지합니다. Teleoperation 충돌 검사는 OFF이며 Initial pose·Reset pose·수동 관절 이동은 ON입니다. 관절 범위·속도·피드백 제한은 유지합니다. 기본 DRY RUN에서 연결과 피드백을 확인한 뒤 REAL RUN 버튼으로 실제 명령 전송을 시작합니다.


In [ ]:
from package.intent_teleop import IntentTeleopSettings, selected_profile, build_models, run_intent_teleop, tracking_mode_text
settings = IntentTeleopSettings(
    backend="real",
    PROFILE_PATH=PROJECT_DIR / "calibration/diagnostics/v1.json",
    POSITION_SCALE=.5, POSITION_SCALE_Y=.5, POSITION_SCALE_Z=.8,
    MAX_JOINT_SPEED=np.deg2rad(60.),  # 실행 창에서 10~120 숫자 입력 + 적용
    TELEOP_RAMP_S=3.0,  # 시작/재그립 후 팔 속도 상한까지 ramp 시간 [초]
    MAX_GRIPPER_SPEED=np.deg2rad(60.),  # 그리퍼 고정 상한 (복귀 중에는 기존 복귀 속도 적용)
    IK_RECOVERY_TIMEOUT_S=2,  # IK 미수렴 시 최대 2초 유지·재시도 (0: 즉시 중단)
    NULL_REFERENCE="self_motion",  # 기존 IK 비교 모드에서만 사용
    POSTURE_IK_ENABLED=True,  # False: 기존 IK로 비교 (런타임 재시작)
    TELEOP_COLLISION_CHECK=False,  # Teleoperation OFF · Initial pose/Reset pose 충돌 검사 ON
    POSTURE_POSITION_TOLERANCE_M=.001,  # IK 후보 허용오차 1 mm
    POSTURE_ROTATION_TOLERANCE_DEG=.5,  # IK 후보 허용오차 0.5°
    POSTURE_ELBOW_LIMIT_DEG=85.,  # 초기 팔꿈치 방향에서 반대편으로 넘어가지 않음
    NULL_ANCHOR_ON_GRIP=False,  # 초기 팔 자세 기준은 첫 조작 시 실측으로 한 번만 저장
    QUEST_HOST=None,  # 자동 LAN IP가 틀리면 PC IP를 문자열로 입력
    TRACKING_PAUSE_S=.1, ACCEPT_EMULATED_POSE=True,  # 고정 모드: Quest 추정 위치 사용 (컨트롤러가 사라지면 0.1초 뒤 정지)
    DRY_RUN=True,
)
settings.validate()
profile, matrices = selected_profile(settings)
print("시작 보정:", "미적용 (identity)" if profile is None else profile["profile_id"])
print("작업자 변경: 실행 창의 작업자 보정 목록에서 파일명을 선택하세요.")
print("IK 모드:", "초기 팔 자세 유지" if settings.POSTURE_IK_ENABLED else "기존 IK")
print("XYZ:", settings.POSITION_SCALES["left"], "| 속도:", np.rad2deg(settings.MAX_JOINT_SPEED),
      "| 그리퍼:", np.rad2deg(settings.MAX_GRIPPER_SPEED), "| 기존 IK null 기준:", settings.NULL_REFERENCE)
print("추적 손실 시작 모드:", tracking_mode_text({"tracking_pause_s": settings.TRACKING_PAUSE_S,
                                                 "accept_emulated": settings.ACCEPT_EMULATED_POSE}))


## 모델 준비 확인 (통신 없음)

기존 FK·IK와 관절 범위 검사를 사용합니다. 충돌 검사는 Teleoperation에서 OFF, Initial pose·Reset pose·수동 조정에서 ON입니다. 아래 셀은 모델만 확인합니다.


In [ ]:
profile, matrices = selected_profile(settings)
models = build_models(settings)
from utils import get_sg2_wrist_control_T
for hand in ("left", "right"):
    print(hand, "EEF (m):", get_sg2_wrist_control_T(models["env"], hand)[:3, 3])
del models


## 실행

[Worker 실행 안내](../docs/WORKER_SETUP.md)에 따라 bringup/outbound/inbound를 실행하세요.

- 다른 Quest 수신 노트북을 STOP하여 8443 포트를 비우세요.
- 양손 grip으로 시작하고, 한쪽이라도 놓으면 양팔이 멈춥니다.
- XYZ는 숫자 입력 후 적용합니다. DRY RUN 대기 중에는 추적 없이 변경할 수 있고, REAL RUN에서는 양손 추적·grip 해제를 확인합니다. DRY RUN에서 바꾼 값은 REAL RUN 전환 후에도 유지되며, 조작 재개는 기존 grip 절차를 따릅니다.
- 속도는 실행 중 입력 후 적용할 수 있습니다. 감속 시 이전 명령 ACK를 기다릴 수 있습니다.
- Initial pose는 Quest/그립 없이 고정된 `worker_bringup` 자세(`BRINGUP_POSE`)로 10°/s 복귀합니다 — 이번 실행에서 로봇이 어떤 자세로 있었는지와 무관합니다. 머리 `head_joint1`의 목표는 bringup과 동일한 0.69 rad(약 39.5°) 아래 방향으로 고정하고, 좌우 회전(`head_joint2`)만 실측값을 유지합니다.
- 누르는 순간 복귀 전체 경로를 기존 관절·5mm 충돌 검사로 확인합니다. 직선 경로가 막히면 J3·J5 먼저 풀기 → 전완·손목 먼저 풀기 → 오른팔 먼저 → 왼팔 먼저 순서로 단계 경로를 찾고, 명령은 확인한 경로 위로만 보냅니다. 모든 경로가 막히면 아무 명령도 보내지 않으므로, VR 조작으로 팔 사이를 벌린 뒤 다시 누르세요. 중간에 멈춘 복귀도 다시 누르면 현재 자세에서 새로 검사합니다.
- **추적 손실 모드**: `추정 위치 사용`으로 고정합니다. 한 손의 광학 추적이 끊겨도 Quest 추정 위치로 조작을 이어가고, 컨트롤러가 아예 사라지면 0.1초 뒤 정지합니다. 추정 위치는 손실 순간 수 cm~십수 cm 튈 수 있습니다.
- Quest 추정 위치 모드에서 한 프레임 20cm 초과 점프·grip 해제·피드백/실행 오류는 정지하고, grip당 20cm 작업범위와 속도 제한은 유지합니다. Quest 추정 위치 모드에서 Teleoperation 충돌 검사는 OFF이며 자동 복귀에서는 ON입니다.
- 일시적인 IK 미수렴은 양팔·그리퍼의 현재 명령을 최대 1초 유지하며 최신 목표로 재시도합니다. 복구되면 같은 grip 기준에서 조작을 이어갑니다. 화면에 경과 시간과 재시도 횟수가 표시되며, 시간 초과·그립 해제·피드백 이상은 정지합니다.
- 팔꿈치 양옆의 두 회전(Joint 3·5)이 조작과 무관하게 감기지 않도록, EEF가 볼 수 없는 자세 성분만 시작 자세 쪽으로 유지합니다. 목표 추종에 쓰이는 성분은 기존과 같이 IK가 결정합니다.
- 로그는 `.cache/fixed_quest/recordings/*-intent-real.jsonl`에 기록됩니다.

In [ ]:
profile, matrices = selected_profile(settings)
log_path = run_intent_teleop(settings)
